# 2 · Simple Simulation — fixed wing

The same scenario as `2-simple_sim_iris.ipynb`; only the airframe changes. Read
that one first for the manual, step-by-step mechanism — this notebook uses the
short form throughout (`AutoPlan.from_relative_path`, `GuidedPlan.from_relative_path`,
`SimVehicle.from_relative`), which resolves relative → absolute positions and
the mission file path internally, so there's no separate conversion step.

A **Zephyr** is a fixed-wing (ArduPlane), so:

* **It cannot hover or turn on the spot.** The circuit is 100 m a side; a
  copter-sized square would leave it turning forever without reaching a waypoint.
* **Home heading matters.** It takes off along it, so `base_home` carries a
  heading here.
* **The plan uses plane actions.** `GuidedPlan` builds `PlaneTakeOff`,
  `PlaneGoTo` and `PlaneLand` instead of the copter versions — `model.firmware`
  selects them, nothing else in the notebook changes.
* **Landing is configured.** `autoland_alt` and `autoland_wp_dist` set up the
  approach, which a copter does not need.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()


## Simulation Positions

Only the deltas from the copter notebook: `base_home` carries a `heading` (the
Zephyr takes off along it) and `base_path` is a `Plan.create_square_path`
circuit, `side_len` metres a side.

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)


In [ ]:
base_home = ENUPose(x=5, y=10, z=0, heading=90)
side_len = 100
base_path = Plan.create_square_path(side_len, alt=20, clockwise=False)


## Create Vehicle

In [ ]:
sysid = 1
color = Color.ORANGE
speed = 14
model = Model.ZEPHYR


### Auto Plan 

In [ ]:
auto_plan = AutoPlan.from_relative_path(
    name="simple_auto_plan",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=base_home,
    relative_path=base_path,
    firmware=model.firmware,
    navigation_speed=speed,
)


In [ ]:
auto_plan


In [ ]:
guided_plan = GuidedPlan.from_relative_path(
    name="simple_guided_plan",
    relative_path=base_path,
    enu_origin=enu_origin,
    relative_home=base_home,
    firmware=model.firmware,
    autoland_alt=10,
    wp_margin=30,
    autoland_wp_dist=side_len - 40,
)
guided_plan


In [ ]:
veh = SimVehicle.from_relative(
    model=model,
    sysid=sysid,
    plan=guided_plan,  # auto_plan,  #
    color=color,
    enu_origin=enu_origin,
    relative_home=base_home,
    relative_path=base_path,
)


In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)
gaz.markers.append(origin_gaz)


### QGroundControl

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(name="origin", pos=gra_origin.unpose(), color=Color.WHITE)
qgc.markers.append(origin_qgc)

### No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)

## Oracle — the scenario

Unchanged from the copter notebook: the vehicle is added to the Oracle, which
holds the scenario. See `2-simple_sim_iris.ipynb` for what else it carries.

In [ ]:
orac = Oracle()

orac.add_vehicle(veh)

## Simulator — the machinery

Unchanged too — the same arguments as in the copter notebook. Only the vehicle
handed to it is different.

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,  # novis,  #
    terminals=[SimProcess.LOGIC],
    verbose=1,
)

simulator.preview()

## Run

`simulator.launch()` then `orac.run()`, as in the copter notebook. They are kept
apart so you can watch the visualizer come up before anything flies;
`simulator.run()` does both at once.

In [ ]:
simulator.launch()


In [ ]:
orac.run()


In [ ]:
orac.plot_trajectories();